In [ ]:
INDEX = 4

# Setup

In [ ]:
import json

def read_jsonl(file_path):
    """
    Reads a JSON Lines (.jsonl) file and returns a list of Python dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
    
    Returns:
        list: A list of dictionaries, one per line in the file.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data
benchmark = read_jsonl("../../benchmark/benchmark_archeology.jsonl")
benchmark[INDEX]

In [ ]:
%load_ext autoreload
%autoreload 2
from os import environ
from sys import path

from torch.backends import cudnn

# enforce more deterministic behavior
environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

path.append("..")

from processor.core.interaction_conductor.chat_interface import ChatInterface, ChatInterfaceOutputFormat

In [ ]:
USER_ID = "llm"
DATA_SOURCES = ["archeology"]
INITIAL_PROMPT = benchmark[INDEX]["interactive_initial_prompt"]

In [ ]:
llm_path = "model/weight/qwen3-8b"
embed_model_path = "model/weight/bge-base"
chat_interface = ChatInterface(llm_path, embed_model_path, USER_ID, DATA_SOURCES)

In [ ]:
from processor.core.ir_system.ir_data_model import convert_multi_retriever_results_to_str


def print_format_to_gpt(ci_output: ChatInterfaceOutputFormat):
    system_output = ci_output['system_response']
    state = ci_output['state']
    current_retrieval_results = ci_output['current_retrieval_results']

    print(f"""SYSTEM OUTPUT:
```{system_output}```

STATE:
```{state}```

RETRIEVED DATA BY THE SYSTEM:
```{convert_multi_retriever_results_to_str(current_retrieval_results)}```
""")

In [ ]:
def print_initial_prompt_to_chatgpt(domain: str, question: str, domain_knowledge: list[str]):
    print(f"""You are simulating a domain expert in world cities, roman cities, radiocarbon data, world conflicts, and climate measurement exploration, who is interacting with a data assistant system to explore insights from an enterprise dataset. The system represents your information need as a set of target schemas, representing relevant table(s) for your question, along with a list of SQL statements, which if runs sequentially on the (materialized) target schemas, will result in the answer of your question.

In this scenario, the system already has access to internal environment-related dataset. You (the simulated user) are already somewhat familiar with the topics of the dataset, as it is commonly used in your team or organization. You are not uploading a new dataset or asking about the existence of some dataset. Your task is to gradually explore or refine your information need about some aspect of the data. You do not begin with a precise question; rather, your curiosity evolves based on system responses and your domain expertise.

Here is a possible eventual goal (you do not know this yet, but may arrive at it through exploration):

{question}

Your behavior should reflect the following:
- You are familiar with the domain.
- You explore and refine your question step-by-step depending on the system's ability to surface relevant information.
- You are allowed to be vague, get sidetracked, or go in the wrong direction.
- You will only arrive at the specific question above if the system's output correctly leads you there.

Continue your role as the domain expert. This is the conversation so far (again, provide response as if you are prompting the system directly):

YOU: {INITIAL_PROMPT}""")

# INTERACTION

In [ ]:
print_initial_prompt_to_chatgpt(
    DATA_SOURCES[0],
    benchmark[INDEX]["original_direct_question"],
    []
)

In [ ]:
output = chat_interface.process_user_input(INITIAL_PROMPT)
print_format_to_gpt(output)

In [ ]:
output = chat_interface.process_user_input(
"""Let’s begin narrowing down the Maltese Neolithic data to focus on northernmost samples and their dating.

Please retrieve from the radiocarbon_database_regional table all Neolithic samples from Malta, along with their latitude, longitude, and the calibrated BC date (using Cal. BC 1 sigma). I want to:
	1.	Identify which of these is the northernmost sample.
	2.	Note the corresponding calibrated date.

Once we have that, we can start looking for nearby climate records from climateMeasurements corresponding to that sample’s year."""
)
print_format_to_gpt(output)

In [ ]:
output = chat_interface.process_user_input(
)
print_format_to_gpt(output)

In [ ]:
output = chat_interface.process_user_input(
)
print_format_to_gpt(output)